# 1. Data preparation, prevalence and sample characteristics

**Study:** Glucose calibration, test timing and socioeconomic patterning of fasting-glucose-defined gestational diabetes in the 2022 Bangladesh Demographic and Health Survey (BDHS).

This notebook is step 1 of 5. It builds the analytic sample and produces the descriptive results.

| Section | What it does | Manuscript |
|---|---|---|
| 1 | Reads and links the survey files, derives variables, defines the sample | Methods 2.1–2.2, Fig. 1 |
| 2 | Weighted prevalence of GDM on both glucose scales | Results 3.1 |
| 3 | Characteristics of the women | Table 1 |

**The five notebooks**

1. `01_data` (this notebook)
2. `02_modeling`
3. `03_inequality`
4. `04_validity`
5. `05_bdhs2017`

The results are stored under each cell, so you can read them without running anything. To run the notebooks yourself, use this order: each one reads what the earlier ones saved. All five together take about a minute.

**Before you run**

- The DHS files are not in this repository. Request them from the [DHS Program](https://dhsprogram.com) and place `BDIR81FL.DTA` and `BDPR81FL.DTA` in `data/raw/bdhs2022/`.
- DHS microdata must never be shared. Do not print individual rows (for example `head(women)`) in a notebook you upload: only summaries are printed here.
- These notebooks run the same steps as `analysis/run_all.R`, so they rewrite the same files in `output/` with the same results.

**Two glucose scales**

| Scale | Variable | Meaning |
|---|---|---|
| Released | `sb267g` | Value released by the DHS, equal to `sb267 × 1.11 / 18` |
| Unconverted | `sb267 / 18` | Same reading without the 1.11 whole-blood-to-plasma factor |

GDM is a fasting plasma glucose (FPG) of 5.1–6.9 mmol/L. Every analysis is run on both scales, and neither is treated as primary.

## Setup

This cell finds the project folder, uses the package versions locked by `renv`, and loads the settings and analysis functions. The functions themselves are in `src/`, so each notebook shows the steps of the analysis without repeating their code.

In [1]:
# Find the project folder, wherever the notebook was opened from
root <- normalizePath(getwd(), winslash = "/")
while (!file.exists(file.path(root, "gdm-project.Rproj"))) {
  if (dirname(root) == root) stop("Open this notebook inside the gdm-project folder.")
  root <- dirname(root)
}
setwd(root)

# Use the package versions locked in renv.lock
if (!nzchar(Sys.getenv("RENV_PROJECT"))) {
  Sys.setenv(RENV_CONFIG_NAMESPACES_CHECK = "FALSE")
  source("renv/activate.R")
}
options(width = 120)

# Settings (config/config.yml) and analysis functions (src/)
source(here::here("src", "common", "setup.R"))

## 1. Build the analytic sample

`read_and_merge()` reads the women's file (IR) and the household member file (PR, which holds the biomarkers) and links them by cluster, household and line number. `derive_variables()` creates the outcomes, exposures and covariates.

Functions: `src/common/data_preparation.R`.

In [2]:
# Read both survey files and link each woman to her biomarker record
files <- read_and_merge(cfg)

# Derive the analysis variables (outcomes, wealth, trimester, BMI and others)
women <- derive_variables(files$women, cfg)

Women in IR: 30078 | matched to PR: 30078 


### Participant flow

How many women remain after each selection step. These numbers feed Fig. 1.

In [3]:
flow <- participant_flow(women, cfg)
print(flow)
save_table(flow, "sample_flow")

                                         step n_remaining n_excluded
1                           Women interviewed       30078         NA
2                          Currently pregnant        1719      28359
3 In household selected for biomarker testing         318       1401
4                      Aged 18 years or older         291         27
5          Valid fasting plasma glucose value         276         15
6   Fasted at least 8 hours (analytic sample)         276          0
7      No self-reported pre-existing diabetes         270          6
8       FPG below 7.0 mmol/L (outcome sample)         264          6
9      Complete covariate data (model sample)         264          0


### Checks on the source data

- **Fasting check:** fasting status and the day of the glucose test.
- **Diabetes history:** answers to the question on a previous diagnosis.
- **Glucose scale relation:** shows that `sb267g` equals `sb267 × 1.11 / 18`.
- **Weight variables:** confirms that no separate biomarker weight is released.
- **Report replication:** reproduces the diabetes estimate of the survey report.

In [4]:
save_table(fasting_check(women), "fasting_check")
save_table(diabetes_history_counts(women), "diabetes_history")
save_table(glucose_scale_relation(files$pr), "glucose_scale_relation")
save_table(weight_variables(cfg), "weight_variables")
save_table(dhs_report_replication(files$pr), "dhs_report_replication")

### Save the derived data

Only the variables used later are kept. The files go to `data/derived/bdhs2022/`, which git ignores.

In [5]:
keep <- c("v001", "v002", "v003", "v021", "v022", "weight", "v190", "v213", "v214",
          "shbpbg", "sb236", "fasted", "fpg", "fpg_alt", "pre_dm",
          "gdm", "gdm_alt", "gdm_incl_dm", "fpg_dm_range", "gdm_incl_dm_alt", "fpg_dm_range_alt",
          "tested_on_fasted_day", "hours_ate", "hours_drank", "drank_never", "drank_8h", "gest_month", "age_c", "bmi_measured", "v025",
          "age", "age_group", "age_group_alt", "education", "working", "residence",
          "birth_order", "gest_weeks", "trimester", "bmi", "hypertension",
          "wealth_score", "wealth", "wealth_v190", "analytic")
women <- women[keep]
saveRDS(women, data_file("women"))

# The analytic sample: pregnant women aged 18 or older with a valid fasting test
sample <- women[women$analytic, ]
saveRDS(sample, data_file("analytic_sample"))
cat("Analytic sample:", sum(women$analytic), "women\n")

Analytic sample: 276 women


## 2. Prevalence of GDM

`svy_domain()` builds the survey design: weight `v005 / 1,000,000`, PSU `v021`, strata `v022`. It keeps all PSUs and estimates within the domain of analytic women, so standard errors reflect the full design.

Functions: `src/common/survey_design.R`.

In [6]:
des <- svy_domain(sample, women)

# Each outcome variable with a readable label
outcomes <- c(gdm              = "GDM, FPG 5.1-6.9 mmol/L, released scale",
              fpg_dm_range     = "FPG 7.0 mmol/L or above, released scale",
              gdm_incl_dm      = "FPG 5.1 mmol/L or above, released scale",
              gdm_alt          = "GDM, FPG 5.1-6.9 mmol/L, unconverted scale",
              fpg_dm_range_alt = "FPG 7.0 mmol/L or above, unconverted scale",
              gdm_incl_dm_alt  = "FPG 5.1 mmol/L or above, unconverted scale")

# Weighted prevalence with a logit-transformed 95% CI for each outcome
prevalence <- do.call(rbind, lapply(names(outcomes), function(v) {
  cbind(outcome = v, label = outcomes[[v]], weighted_prevalence(des, v))
}))
save_table(prevalence, "prevalence")

# Easy-to-read view: percentages with their 95% CIs
data.frame(outcome = prevalence$label, n = prevalence$n,
           percent = round(100 * prevalence$prevalence, 1),
           ci_lower = round(100 * prevalence$ci_lower, 1),
           ci_upper = round(100 * prevalence$ci_upper, 1))

                                     outcome   n percent ci_lower ci_upper
1    GDM, FPG 5.1-6.9 mmol/L, released scale 264    63.2     56.3     69.7
2    FPG 7.0 mmol/L or above, released scale 270     2.2      0.9      5.1
3    FPG 5.1 mmol/L or above, released scale 270    64.0     57.2     70.4
4 GDM, FPG 5.1-6.9 mmol/L, unconverted scale 267    32.3     26.4     38.8
5 FPG 7.0 mmol/L or above, unconverted scale 270     0.9      0.2      3.7
6 FPG 5.1 mmol/L or above, unconverted scale 270    32.9     27.0     39.4


### Prevalence by trimester

Surveys test women at whatever stage of pregnancy they are interviewed, so this shows how much the estimate depends on timing.

In [7]:
by_trimester <- do.call(rbind, lapply(names(scales), function(s) {
  cbind(scale = s, prevalence_by(des, scales[[s]], "trimester"))
}))
save_table(by_trimester, "prevalence_by_trimester")
print(by_trimester)

        scale     group level  n events prevalence         se  ci_lower  ci_upper
1    released trimester   1st 78     63  0.7937334 0.05124923 0.6729679 0.8779884
2    released trimester   2nd 97     56  0.6039760 0.05379677 0.4935733 0.7047098
3    released trimester   3rd 89     48  0.5197999 0.05448714 0.4118707 0.6259121
4 unconverted trimester   1st 78     42  0.4843532 0.06205825 0.3636709 0.6068884
5 unconverted trimester   2nd 98     26  0.2855282 0.05309486 0.1922819 0.4015153
6 unconverted trimester   3rd 91     23  0.2230343 0.04654764 0.1438511 0.3290522


### Survey design checks

- **Design effect:** how much the design inflates the variance (Table SA10).
- **PSU structure:** PSUs per stratum, in the full and analytic samples.
- **Design comparison:** standard errors from the domain design against a design on the analytic rows only.

In [8]:
design_effects <- do.call(rbind, lapply(names(scales), function(s) {
  cbind(scale = s, design_effect(des, sample, scales[[s]]))
}))
save_table(design_effects, "design_effect")
print(design_effects)

save_table(psu_structure(sample, women), "psu_structure")
save_table(design_comparison(sample, women, cfg), "design_comparison")

        scale   n prevalence         se     deff deff_weighting deff_clustering effective_n
1    released 264  0.6323521 0.03408295 1.314132       1.190142        1.104181    200.8931
2 unconverted 267  0.3227938 0.03144120 1.202912       1.187697        1.012810    221.9614


## 3. Sample characteristics (Table 1)

Categorical and continuous characteristics, with GDM prevalence by characteristic on both scales, and the same characteristics by household wealth tertile.

In [9]:
vars <- c("age_group", "education", "working", "residence", "wealth", "birth_order",
          "trimester", "hypertension")

# GDM by characteristic, on each scale
t1_cat  <- do.call(rbind, lapply(scales, function(o) describe_categorical(des, vars, o)))
t1_cont <- do.call(rbind, lapply(scales, function(o) describe_continuous(des, c("age", "gest_weeks", "bmi"), o)))
print(t1_cat)
save_table(t1_cat, "table1_categorical")
save_table(t1_cont, "table1_continuous")
save_table(residence_by_wealth(des), "residence_by_wealth")

               outcome     variable            level   n n_gdm pct_of_sample  pct_gdm n_missing      p_value
released.1         gdm    age_group              <25 148    87     57.545484 59.04494         0 1.534599e-01
released.2         gdm    age_group              25+ 116    80     42.454516 68.91496         0 1.534599e-01
released.3         gdm    education Primary or below  52    32     18.317906 58.52201         0 6.460947e-01
released.4         gdm    education        Secondary 139    86     55.368855 62.74292         0 6.460947e-01
released.5         gdm    education           Higher  73    49     26.313239 67.55217         0 6.460947e-01
released.6         gdm      working               No 215   131     80.078744 60.89082         0 1.581168e-01
released.7         gdm      working              Yes  49    36     19.921256 72.65909         0 1.581168e-01
released.8         gdm    residence            Rural 187   111     77.854462 58.93953         0 4.916904e-03
released.9         

In [10]:
# Table 1 of the manuscript: characteristics by household wealth tertile
t1_wealth <- describe_by_wealth(des, vars, c("age", "gest_weeks", "bmi"), as.list(scales))
print(t1_wealth)
save_table(t1_wealth, "table1_by_wealth")

          variable            level   group   n     value        sd      p_value
1        age_group              <25   Total 148 57.545484        NA 1.985771e-01
2        age_group              25+   Total 116 42.454516        NA 1.985771e-01
3        age_group              <25  Lowest  55 64.769693        NA 1.985771e-01
4        age_group              25+  Lowest  33 35.230307        NA 1.985771e-01
5        age_group              <25  Middle  50 57.275843        NA 1.985771e-01
6        age_group              25+  Middle  40 42.724157        NA 1.985771e-01
7        age_group              <25 Highest  43 49.581179        NA 1.985771e-01
8        age_group              25+ Highest  43 50.418821        NA 1.985771e-01
9        education Primary or below   Total  52 18.317906        NA 6.062320e-09
10       education        Secondary   Total 139 55.368855        NA 6.062320e-09
11       education           Higher   Total  73 26.313239        NA 6.062320e-09
12       education Primary o

**Next:** `02_modeling`.